In [ ]:
# Check GPU
import torch
print("PyTorch Version ",torch.__version__)
print("GPU Available ",torch.cuda.is_available())
if torch.cuda.is_available():
    print("GPU Name ",torch.cuda.get_device_name(0))

PyTorch Version  2.11.0+cu128
GPU Available  True
GPU Name  Tesla T4


In [ ]:
!pip install -q transformers datasets sentencepiece accelerate evaluate rouge_score gradio kagglehub

  Preparing metadata (setup.py) ... done
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 84.1/84.1 kB 5.2 MB/s eta 0:00:00


In [ ]:
import kagglehub

path = kagglehub.dataset_download(
    "snap/amazon-fine-food-reviews"
)

print("Dataset downloaded at:")
print(path)

100%|██████████| 242M/242M [00:01<00:00, 150MB/s]

Extracting files...


Dataset downloaded at:
/root/.cache/kagglehub/datasets/snap/amazon-fine-food-reviews/versions/2


In [ ]:
#load dataset
import pandas as pd
import os
print(os.listdir(path))


['Reviews.csv', 'hashes.txt', 'database.sqlite']


In [ ]:
#CSV load
df=pd.read_csv(os.path.join(path,"Reviews.csv"))
df

,Id,ProductId,UserId,ProfileName,HelpfulnessNumerator,HelpfulnessDenominator,Score,Time,Summary,Text
0,1,B001E4KFG0,A3SGXH7AUHU8GW,delmartian,1,1,5,1303862400,Good Quality Dog Food,I have bought several of the Vitality canned d...
1,2,B00813GRG4,A1D87F6ZCVE5NK,dll pa,0,0,1,1346976000,Not as Advertised,Product arrived labeled as Jumbo Salted Peanut...
2,3,B000LQOCH0,ABXLMWJIXXAIN,"Natalia Corres ""Natalia Corres""",1,1,4,1219017600,"""Delight"" says it all",This is a confection that has been around a fe...
3,4,B000UA0QIQ,A395BORC6FGVXV,Karl,3,3,2,1307923200,Cough Medicine,If you are looking for the secret ingredient i...
4,5,B006K2ZZ7K,A1UQRSCLF8GW1T,"Michael D. Bigham ""M. Wassir""",0,0,5,1350777600,Great taffy,Great taffy at a great price. There was a wid...
...,...,...,...,...,...,...,...,...,...,...
568449,568450,B001EO7N10,A28KG5XORO54AY,Lettie D. Carter,0,0,5,1299628800,Will not do without,Great for sesame chicken..this is a good if no...
568450,568451,B003S1WTCU,A3I8AFVPEE8KI5,R. Sawyer,0,0,2,1331251200,disappointed,I'm disappointed with the flavor. The chocolat...
568451,568452,B004I613EE,A121AA1GQV751Z,"pksd ""pk_007""",2,2,5,1329782400,Perfect for our maltipoo,"These stars are small, so you can give 10-15 o..."
568452,568453,B004I613EE,A3IBEVCTXKNOH,"Kathy A. Welch ""katwel""",1,1,5,1331596800,Favorite Training and reward treat,These are the BEST treats for training and rew...


In [ ]:
df=df[['Text','Summary']]
df

,Text,Summary
0,I have bought several of the Vitality canned d...,Good Quality Dog Food
1,Product arrived labeled as Jumbo Salted Peanut...,Not as Advertised
2,This is a confection that has been around a fe...,"""Delight"" says it all"
3,If you are looking for the secret ingredient i...,Cough Medicine
4,Great taffy at a great price. There was a wid...,Great taffy
...,...,...
568449,Great for sesame chicken..this is a good if no...,Will not do without
568450,I'm disappointed with the flavor. The chocolat...,disappointed
568451,"These stars are small, so you can give 10-15 o...",Perfect for our maltipoo
568452,These are the BEST treats for training and rew...,Favorite Training and reward treat


In [ ]:
#Remove missing values
df=df.dropna()
df=df[(df['Text'].str.len()>20) & (df['Summary'].str.len()>2)]
df


,Text,Summary
0,I have bought several of the Vitality canned d...,Good Quality Dog Food
1,Product arrived labeled as Jumbo Salted Peanut...,Not as Advertised
2,This is a confection that has been around a fe...,"""Delight"" says it all"
3,If you are looking for the secret ingredient i...,Cough Medicine
4,Great taffy at a great price. There was a wid...,Great taffy
...,...,...
568449,Great for sesame chicken..this is a good if no...,Will not do without
568450,I'm disappointed with the flavor. The chocolat...,disappointed
568451,"These stars are small, so you can give 10-15 o...",Perfect for our maltipoo
568452,These are the BEST treats for training and rew...,Favorite Training and reward treat


In [ ]:
# Take small Dataset
df=df.sample(
    10000,random_state=42
).reset_index(drop=True)
print(df.shape)


(10000, 2)


In [ ]:
# Train/Test split
from sklearn.model_selection import train_test_split
train_df,test_df=train_test_split(
    df,test_size=0.2,random_state=42
)

In [ ]:
#Convert to Hugging Face Dataset
from datasets import Dataset
train_dataset=Dataset.from_pandas(train_df)
test_dataset=Dataset.from_pandas(test_df)

In [ ]:
#load T5 tokenizer
from transformers import AutoTokenizer
model_name='t5-small'
tokenizer=AutoTokenizer.from_pretrained(model_name)
print(tokenizer)

config.json:   0%|          | 0.00/1.21k [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/2.32k [00:00<?, ?B/s]

spiece.model:   0%|          | 0.00/792k [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/1.39M [00:00<?, ?B/s]

T5Tokenizer(name_or_path='t5-small', vocab_size=32100, model_max_length=512, padding_side='right', truncation_side='right', special_tokens={'eos_token': '</s>', 'unk_token': '<unk>', 'pad_token': '<pad>'}, added_tokens_decoder={
	0: AddedToken("<pad>", rstrip=False, lstrip=False, single_word=False, normalized=False, special=True),
	1: AddedToken("</s>", rstrip=False, lstrip=False, single_word=False, normalized=False, special=True),
	2: AddedToken("<unk>", rstrip=False, lstrip=False, single_word=False, normalized=False, special=True),
	32000: AddedToken("<extra_id_99>", rstrip=False, lstrip=False, single_word=False, normalized=False, special=True),
	32001: AddedToken("<extra_id_98>", rstrip=False, lstrip=False, single_word=False, normalized=False, special=True),
	32002: AddedToken("<extra_id_97>", rstrip=False, lstrip=False, single_word=False, normalized=False, special=True),
	32003: AddedToken("<extra_id_96>", rstrip=False, lstrip=False, single_word=False, normalized=False, special=Tru

In [ ]:
sample_text = train_df.iloc[0]["Text"]

print("Original:")
print(sample_text)

tokens = tokenizer(
    sample_text,
    return_tensors="pt"
)

print("\nInput IDs:")
print(tokens["input_ids"])

print("\nTokens:")
print(tokenizer.convert_ids_to_tokens(
    tokens["input_ids"][0]
))

Original:
I love herbal tea, especially Celestial Seasonings..  The Mandarin Orange Spice is one of my favorites.  It tastes great both hot and cold and is very refreshing. Buying it at Amazon is the best deal (save $5.00 as compared to markets).

Input IDs:
tensor([[   27,   333, 18543,  2198,     6,   902, 13136,     7, 10646,  7960,
            53,     7,     5,     5,    37, 31057,  7311, 25727,    19,    80,
            13,    82, 13127,     5,    94, 13434,   248,   321,  1312,    11,
          2107,    11,    19,   182, 13132,     5,     3, 22522,    34,    44,
          2536,    19,     8,   200,  1154,    41,     7,     9,   162,  6422,
          4200,    38,     3,  2172,    12,  3212,   137,     1]])

Tokens:
['▁I', '▁love', '▁herbal', '▁tea', ',', '▁especially', '▁Cele', 's', 'tial', '▁Season', 'ing', 's', '.', '.', '▁The', '▁Mandarin', '▁Orange', '▁Spice', '▁is', '▁one', '▁of', '▁my', '▁favorites', '.', '▁It', '▁tastes', '▁great', '▁both', '▁hot', '▁and', '▁cold', '▁and', 

In [ ]:
#Prepare Input and Target
def preprocess_function(examples):

    inputs = [
        "summarize: " + text
        for text in examples["Text"]
    ]

    targets = examples["Summary"]

    model_inputs = tokenizer(
        inputs,
        max_length=256,
        truncation=True
    )

    labels = tokenizer(
        text_target=targets,
        max_length=64,
        truncation=True
    )

    model_inputs["labels"] = labels["input_ids"]

    return model_inputs

In [ ]:
tokenized_train=train_dataset.map(
    preprocess_function,
    batched=True
)
tokenized_test=test_dataset.map(
    preprocess_function,
    batched=True
)

Map:   0%|          | 0/8000 [00:00<?, ? examples/s]

Map:   0%|          | 0/2000 [00:00<?, ? examples/s]

In [ ]:
tokenized_train[0]

{'Text': 'I love herbal tea, especially Celestial Seasonings..  The Mandarin Orange Spice is one of my favorites.  It tastes great both hot and cold and is very refreshing. Buying it at Amazon is the best deal (save $5.00 as compared to markets).',
 'Summary': 'Best Bargain in Town',
 '__index_level_0__': 9254,
 'input_ids': [21603,
  10,
  27,
  333,
  18543,
  2198,
  6,
  902,
  13136,
  7,
  10646,
  7960,
  53,
  7,
  5,
  5,
  37,
  31057,
  7311,
  25727,
  19,
  80,
  13,
  82,
  13127,
  5,
  94,
  13434,
  248,
  321,
  1312,
  11,
  2107,
  11,
  19,
  182,
  13132,
  5,
  3,
  22522,
  34,
  44,
  2536,
  19,
  8,
  200,
  1154,
  41,
  7,
  9,
  162,
  6422,
  4200,
  38,
  3,
  2172,
  12,
  3212,
  137,
  1],
 'attention_mask': [1,
  1,
  1,
  1,
  1,
  1,
  1,
  1,
  1,
  1,
  1,
  1,
  1,
  1,
  1,
  1,
  1,
  1,
  1,
  1,
  1,
  1,
  1,
  1,
  1,
  1,
  1,
  1,
  1,
  1,
  1,
  1,
  1,
  1,
  1,
  1,
  1,
  1,
  1,
  1,
  1,
  1,
  1,
  1,
  1,
  1,
  1,
  1,
  1,
  1

In [ ]:
#load Transformer model
from transformers import AutoModelForSeq2SeqLM
model=AutoModelForSeq2SeqLM.from_pretrained(model_name)

model.safetensors: reconstructing file:   0%|          |  0.00B /  242MB            

model.safetensors: downloading bytes:           |  0.00B            

Loading weights:   0%|          | 0/131 [00:00<?, ?it/s]

generation_config.json:   0%|          | 0.00/147 [00:00<?, ?B/s]

In [ ]:
print(model)

T5ForConditionalGeneration(
  (shared): Embedding(32128, 512)
  (encoder): T5Stack(
    (embed_tokens): Embedding(32128, 512)
    (block): ModuleList(
      (0): T5Block(
        (layer): ModuleList(
          (0): T5LayerSelfAttention(
            (SelfAttention): T5Attention(
              (q): Linear(in_features=512, out_features=512, bias=False)
              (k): Linear(in_features=512, out_features=512, bias=False)
              (v): Linear(in_features=512, out_features=512, bias=False)
              (o): Linear(in_features=512, out_features=512, bias=False)
              (relative_attention_bias): Embedding(32, 8)
            )
            (layer_norm): T5LayerNorm()
            (dropout): Dropout(p=0.1, inplace=False)
          )
          (1): T5LayerFF(
            (DenseReluDense): T5DenseActDense(
              (wi): Linear(in_features=512, out_features=2048, bias=False)
              (wo): Linear(in_features=2048, out_features=512, bias=False)
              (dropout): Drop

In [ ]:
#check encoder output
text='The food was nice and very testy'
inputs=tokenizer(text,return_tensors='pt')
with torch.no_grad():

    encoder_outputs = model.encoder(
        input_ids=inputs["input_ids"],
        attention_mask=inputs["attention_mask"]
    )

print("Encoder output shape:")
print(encoder_outputs.last_hidden_state.shape)

Encoder output shape:
torch.Size([1, 9, 512])


In [ ]:
device = "cuda" if torch.cuda.is_available() else "cpu"

model = model.to(device)

text = """
The food was excellent. The taste was amazing and
the packaging was very good. I really enjoyed this
product and would definitely buy it again.
"""

inputs = tokenizer(
    "summarize: " + text,
    return_tensors="pt",
    truncation=True,
    max_length=256
).to(device)

outputs = model.generate(
    **inputs,
    max_length=50,
    num_beams=4,
    early_stopping=True
)

summary = tokenizer.decode(
    outputs[0],
    skip_special_tokens=True
)

print("Original:")
print(text)

print("\nGenerated Summary:")
print(summary)

Original:

The food was excellent. The taste was amazing and
the packaging was very good. I really enjoyed this
product and would definitely buy it again.


Generated Summary:
the food was excellent. The taste was amazing and the packaging was very good.
